# Neural Networks for Sequences — Exercises — STUDENT VERSION

These exercises accompany the lecture slides *"Neural Networks for Sequences"* (K. Reluga) and let you implement, in Python, three of the main ideas covered there:

1. **Sequence classification (seq2vec) with an RNN** — using the final hidden state of an `nn.LSTM` as input to a classifier
2. **Scaled dot-product attention** — implementing the core attention formula from scratch and checking it against PyTorch's built-in version
3. **Vec2Seq sequence generation** — using `nn.RNNCell` to generate a sequence one token at a time from a single input vector, trained with teacher forcing

Each exercise has a short recap of the relevant slide content, a code cell with `TODO`s for you to fill in, and a test/demo cell that uses your implementation.

**Note on compute**: all three exercises are small and synthetic (no external datasets or pretrained weights to download), and run in well under a minute on a CPU.


## 0. Setup: required libraries

Before running this notebook, make sure the following libraries are installed:

- `numpy`
- `matplotlib`
- `torch` (PyTorch)

You can install them all at once with:

```bash
pip install numpy matplotlib torch
```

(No GPU or dataset/weight downloads are required for this notebook — everything runs on small, synthetically generated data.)

The cell below **checks** which of these libraries are already available in your Python environment, and prints the exact `pip install` command for anything that is missing.


In [ ]:
import importlib

required_libraries = {
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "torch": "torch",
}

missing = []
for import_name, pip_name in required_libraries.items():
    try:
        importlib.import_module(import_name)
        print(f"[OK]      {import_name:<10s} is installed.")
    except ImportError:
        print(f"[MISSING] {import_name:<10s} is NOT installed.")
        missing.append(pip_name)

print()
if missing:
    print("Some libraries are missing. Install them by running this command")
    print("in a terminal (or in a notebook cell, prefixed with '!'):\n")
    print(f"    pip install {' '.join(missing)}")
else:
    print("All required libraries are installed -- you are ready to go!")


## Common imports

Run this cell once at the start; all exercises rely on it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

%matplotlib inline
np.random.seed(0)
torch.manual_seed(0)


---
## Exercise 1 — Sequence classification (seq2vec) with an RNN

**Recap (slides):** in sequence classification (**seq2vec**), we want a single fixed-length output $y$ given a variable-length input sequence $\boldsymbol{x}_{1:T}$. The simplest approach uses the **final hidden state** $\boldsymbol{h}_T$ of an RNN as input to a classifier:

$$p(y\mid\boldsymbol{x}_{1:T}) = \text{Cat}\big(y \mid \text{softmax}(\mathbf{W}\boldsymbol{h}_T)\big)$$

**Task:** we classify short sequences of digits (integers $0,\dots,9$): the label is $1$ if the digit $5$ appears **anywhere** in the sequence, and $0$ otherwise. This requires the RNN to carry information forward in its hidden state until the end of the sequence — exactly what the seq2vec architecture above is designed to do.

**Your task:** complete `Seq2VecRNN`, which should:
1. embed the input tokens (`nn.Embedding`)
2. run them through an `nn.LSTM` (`batch_first=True`)
3. take the LSTM's **final hidden state** and map it to class logits with a linear layer


In [ ]:
class Seq2VecRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        # TODO: define
        #   self.embedding = nn.Embedding(vocab_size, embed_dim)
        #   self.lstm       = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        #   self.classifier = nn.Linear(hidden_dim, num_classes)
        raise NotImplementedError("Define the layers")

    def forward(self, x):
        # x: (batch, seq_len) of token indices
        # TODO 1: embed x -> shape (batch, seq_len, embed_dim)
        # TODO 2: run through self.lstm; it returns
        #         output, (h_n, c_n), where h_n has shape
        #         (num_layers, batch, hidden_dim)
        # TODO 3: take the last layer's final hidden state h_n[-1]
        #         (shape (batch, hidden_dim)) and pass it through
        #         self.classifier to get the (batch, num_classes) logits
        raise NotImplementedError("Implement the forward pass")


**Data, training and evaluation** (given): we generate random digit sequences and label them by whether they contain a `5`, train `Seq2VecRNN` for a few epochs, and report test accuracy (a well-trained model should reach well above the 50% random baseline).

In [ ]:
def make_contains_digit_dataset(n_samples, seq_len, vocab_size=10, target_digit=5):
    X = np.random.randint(0, vocab_size, size=(n_samples, seq_len))
    y = (X == target_digit).any(axis=1).astype(np.int64)
    return torch.tensor(X, dtype=torch.long), torch.tensor(y, dtype=torch.long)

X_train, y_train = make_contains_digit_dataset(2000, seq_len=10)
X_test, y_test = make_contains_digit_dataset(400, seq_len=10)
print("Fraction of positive (label=1) examples in training set:", y_train.float().mean().item())

model = Seq2VecRNN(vocab_size=10, embed_dim=8, hidden_dim=16, num_classes=2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
criterion = nn.CrossEntropyLoss()

n_epochs = 15
batch_size = 64
for epoch in range(n_epochs):
    model.train()
    perm = torch.randperm(len(X_train))
    total_loss = 0.0
    for i in range(0, len(X_train), batch_size):
        idx = perm[i:i + batch_size]
        xb, yb = X_train[idx], y_train[idx]
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(idx)
    if (epoch + 1) % 5 == 0:
        print(f"epoch {epoch + 1:2d}/{n_epochs} - train loss: {total_loss / len(X_train):.4f}")

model.eval()
with torch.no_grad():
    preds = model(X_test).argmax(dim=1)
    acc = (preds == y_test).float().mean().item()
print(f"\nTest accuracy: {acc * 100:.2f}%  (random baseline: 50%)")


---
## Exercise 2 — Scaled dot-product attention from scratch

**Recap (slides):** attention retrieves a weighted combination of **values** $\mathbf{V}$, where the weights come from comparing **queries** $\mathbf{Q}$ to **keys** $\mathbf{K}$. For queries/keys of dimension $d$, **scaled dot-product attention** is

$$\text{Attn}(\mathbf{Q},\mathbf{K},\mathbf{V}) = \text{softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d}}\right)\mathbf{V}$$

where the softmax is applied **row-wise** (i.e. separately for each query, over all keys), so each row of attention weights sums to 1.

**Your task:** implement `scaled_dot_product_attention(Q, K, V)` using plain NumPy.


In [ ]:
def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)


def scaled_dot_product_attention(Q, K, V):
    """
    Q: (n, d) query matrix
    K: (m, d) key matrix
    V: (m, v) value matrix

    Returns
    -------
    output  : (n, v) attention output
    weights : (n, m) attention weight matrix (rows sum to 1)
    """
    # TODO 1: compute the (n, m) matrix of scaled dot-product scores,
    #         Q @ K.T / sqrt(d), where d is the query/key dimension
    #         (Q.shape[-1])

    # TODO 2: apply the `softmax` function above (along the last axis) to
    #         turn the scores into attention weights

    # TODO 3: compute the output as weights @ V

    raise NotImplementedError("Implement scaled_dot_product_attention")


**Test / demo** (given): we run your function on small random $\mathbf{Q}, \mathbf{K}, \mathbf{V}$ matrices, check the output/weight shapes and that weight rows sum to 1, compare the result against PyTorch's built-in `torch.nn.functional.scaled_dot_product_attention`, and visualize the attention weight matrix as a heat map (as in the *"kernel regression as attention"* slide).

In [ ]:
n, m, d, v = 5, 7, 4, 3   # n queries, m keys/values, dim d for Q/K, dim v for V
Q = np.random.randn(n, d)
K = np.random.randn(m, d)
V = np.random.randn(m, v)

output, weights = scaled_dot_product_attention(Q, K, V)
print("output shape :", output.shape, " (expected", (n, v), ")")
print("weights shape:", weights.shape, " (expected", (n, m), ")")
print("rows of weights sum to 1:", np.allclose(weights.sum(axis=1), 1.0))

# Compare against PyTorch's built-in scaled dot-product attention
Qt = torch.tensor(Q).unsqueeze(0)
Kt = torch.tensor(K).unsqueeze(0)
Vt = torch.tensor(V).unsqueeze(0)
ref_output = F.scaled_dot_product_attention(Qt, Kt, Vt).squeeze(0).numpy()
print("Matches torch.nn.functional.scaled_dot_product_attention:",
      np.allclose(output, ref_output, atol=1e-6))

plt.figure(figsize=(5, 4))
plt.imshow(weights, cmap="viridis", aspect="auto")
plt.colorbar(label="attention weight")
plt.xlabel("key index")
plt.ylabel("query index")
plt.title("Attention weight matrix")
plt.show()


---
## Exercise 3 — Vec2Seq: generating a sequence from a single vector

**Recap (slides):** a **vec2seq** model learns $f_{\boldsymbol{\theta}}:\mathbb{R}^D \to \mathbb{R}^{N_\infty C}$, mapping a single input vector $\boldsymbol{x}$ to an output *sequence* $\boldsymbol{y}_{1:T}$, generated **one token at a time**:

- the initial hidden state is computed directly from the input: $p(\boldsymbol{h}_1\mid\boldsymbol{h}_0,\boldsymbol{y}_0,\boldsymbol{x}) = p(\boldsymbol{h}_1\mid\boldsymbol{x})$
- at every later step, the hidden state update is $\boldsymbol{h}_t = \varphi(\mathbf{W}_{xh}[\boldsymbol{x};\boldsymbol{y}_{t-1}] + \mathbf{W}_{hh}\boldsymbol{h}_{t-1} + \boldsymbol{b}_h)$
- at every step, the output is sampled from the hidden state: $p(\boldsymbol{y}_t\mid\boldsymbol{h}_t) = \text{Cat}(\boldsymbol{y}_t\mid\text{softmax}(\mathbf{W}_{hy}\boldsymbol{h}_t + \boldsymbol{b}_y))$
- to **generate**, we feed the sampled/predicted $\boldsymbol{y}_t$ back in to compute $\boldsymbol{h}_{t+1}$, and repeat

**Task:** given a starting digit $s \in \{0,\dots,9\}$ as a one-hot vector $\boldsymbol{x}$, generate the "counting" sequence $y_t = (s+t) \bmod 10$ for $t=0,\dots,T-1$ (e.g. $s=7,\,T=5 \Rightarrow 7,8,9,0,1$). This is a simple, fully deterministic instance of the vec2seq architecture above, built from two existing PyTorch layers: `nn.Linear` (to get $\boldsymbol{h}_1$ from $\boldsymbol{x}$) and `nn.RNNCell` (to implement the per-step update $\boldsymbol{h}_t = \varphi(\dots)$).

**Your task:** complete the two methods of `Vec2SeqRNN`:
1. `init_state(x)`: turn the one-hot conditioning vector $\boldsymbol{x}$ into the initial hidden state $\boldsymbol{h}_1$
2. `step(y_prev, h_prev)`: one recurrent step — embed the previous token, update the hidden state with `nn.RNNCell`, and compute this step's output logits


In [ ]:
class Vec2SeqRNN(nn.Module):
    def __init__(self, num_classes, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_classes, hidden_dim)   # embeds y_{t-1}
        self.init_hidden = nn.Linear(num_classes, hidden_dim)    # x -> h_1
        self.rnn_cell = nn.RNNCell(hidden_dim, hidden_dim)       # (embedded y_{t-1}, h_{t-1}) -> h_t
        self.output_layer = nn.Linear(hidden_dim, num_classes)   # h_t -> logits over next token

    def init_state(self, x):
        """x: (batch, num_classes) one-hot conditioning vector -> h_1: (batch, hidden_dim)"""
        # TODO: compute and return the initial hidden state h_1 from x.
        #       Use self.init_hidden(x) followed by a torch.tanh nonlinearity.
        raise NotImplementedError("Implement init_state")

    def step(self, y_prev, h_prev):
        """
        y_prev : (batch,) LongTensor, previous token
        h_prev : (batch, hidden_dim) previous hidden state
        Returns (logits, h_t): logits over the next token, and the new hidden state
        """
        # TODO 1: embed y_prev with self.embedding -> shape (batch, hidden_dim)
        # TODO 2: update the hidden state: h_t = self.rnn_cell(embedded, h_prev)
        # TODO 3: compute logits = self.output_layer(h_t)
        # return logits, h_t
        raise NotImplementedError("Implement step")


**Training (teacher forcing) and generation** (given): we train with **teacher forcing** — at each step the *ground-truth* previous token is fed into `step`, following the *"Teacher Forcing"* slide — summing the cross-entropy loss over all $T$ steps. We then **generate** sequences autoregressively: starting only from $\boldsymbol{x}$, at each step we take the model's own predicted token and feed it back in to compute the next hidden state, exactly as described in the *"Vec2Seq: Generation Process"* slide.

In [ ]:
num_classes = 10   # digits 0-9
hidden_dim = 16
T = 5              # sequence length

def make_counting_dataset(n_samples, T, num_classes=10):
    starts = np.random.randint(0, num_classes, size=n_samples)
    X = np.eye(num_classes, dtype=np.float32)[starts]                      # one-hot conditioning vector
    Y = np.stack([(starts + t) % num_classes for t in range(T)], axis=1)   # (n_samples, T)
    return torch.tensor(X), torch.tensor(Y, dtype=torch.long)

X_train, Y_train = make_counting_dataset(2000, T)
X_test, Y_test = make_counting_dataset(200, T)

model = Vec2SeqRNN(num_classes=num_classes, hidden_dim=hidden_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
criterion = nn.CrossEntropyLoss()

def teacher_forced_loss(model, x, y):
    # h_1 comes directly from x; the first token is predicted straight from h_1
    h = model.init_state(x)
    logits_1 = model.output_layer(h)
    loss = criterion(logits_1, y[:, 0])
    # subsequent steps: teacher forcing feeds the *ground-truth* previous token
    for t in range(1, y.shape[1]):
        logits_t, h = model.step(y[:, t - 1], h)
        loss = loss + criterion(logits_t, y[:, t])
    return loss / y.shape[1]

n_epochs = 100
batch_size = 64
for epoch in range(n_epochs):
    model.train()
    perm = torch.randperm(len(X_train))
    total_loss = 0.0
    for i in range(0, len(X_train), batch_size):
        idx = perm[i:i + batch_size]
        xb, yb = X_train[idx], Y_train[idx]
        optimizer.zero_grad()
        loss = teacher_forced_loss(model, xb, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(idx)
    if (epoch + 1) % 20 == 0:
        print(f"epoch {epoch + 1:3d}/{n_epochs} - train loss: {total_loss / len(X_train):.4f}")

@torch.no_grad()
def generate(model, x, T):
    """Autoregressive vec2seq generation: feed each predicted token back in."""
    model.eval()
    h = model.init_state(x)
    logits = model.output_layer(h)
    y_t = logits.argmax(dim=-1)
    ys = [y_t]
    for _ in range(1, T):
        logits, h = model.step(y_t, h)
        y_t = logits.argmax(dim=-1)
        ys.append(y_t)
    return torch.stack(ys, dim=1)   # (batch, T)

generated = generate(model, X_test, T)
accuracy = (generated == Y_test).float().mean().item()
print(f"\nPer-token accuracy on generated test sequences: {accuracy * 100:.2f}%")

print("\nA few examples (starting digit -> generated sequence vs. true sequence):")
starts = X_test.argmax(dim=1)
for i in range(5):
    print(f"  start={starts[i].item()}: generated={generated[i].tolist()}  true={Y_test[i].tolist()}")
